# FASE 4 - Reporte Automatizado: Alertas de Pacientes Criticos
## Proyecto Cardiologia Analytics

Simula un proceso diario (ej. 06:00 AM) que lee pacientes nuevos, corre el modelo
de la Fase 2, filtra los criticos (riesgo > 80%) y genera un correo HTML para la
Jefatura de Cardiologia. **Modo simulado:** el HTML se guarda localmente, no se envia.

Programacion real (cron, no incluida): `0 6 * * * python3 fase4_alertas_criticas.py`


### Celda 1 - Librerias y configuracion

In [ ]:
import os
from datetime import datetime
import pandas as pd
import joblib
from IPython.display import HTML, display

UMBRAL_CRITICO = 0.80
DESTINATARIO = "Jefatura de Cardiologia"
ASUNTO = "URGENTE: Alerta de Pacientes Criticos Detectados en las Ultimas 24h"
NUMERICAS   = ["Age","RestingBP","Cholesterol","FastingBS","MaxHR","Oldpeak"]
CATEGORICAS = ["Sex","ChestPainType","RestingECG","ExerciseAngina","ST_Slope"]
print("Configuracion lista. Umbral critico:", UMBRAL_CRITICO)

### Celda 2 - Cargar pacientes nuevos y el modelo de la Fase 2

In [ ]:
df = pd.read_csv("../data/nuevos_pacientes.csv")
modelo = joblib.load("../output/modelo_rf.joblib")
print("Pacientes nuevos:", len(df))
df.head()

### Celda 3 - Predecir riesgo y filtrar criticos (> 80%)

In [ ]:
X = df[NUMERICAS + CATEGORICAS]
df["Probabilidad_Riesgo"] = modelo.predict_proba(X)[:, 1]
df["ID_Paciente"] = ["NUEVO-" + str(i).zfill(3) for i in range(1, len(df)+1)]

criticos = df[df["Probabilidad_Riesgo"] > UMBRAL_CRITICO].sort_values(
    "Probabilidad_Riesgo", ascending=False).copy()
print("Criticos detectados:", len(criticos))
criticos[["ID_Paciente","Age","Sex","ChestPainType","ST_Slope","Probabilidad_Riesgo"]]

### Celda 4 - Funciones de apoyo (accion sugerida y construccion del HTML)
Reusa la logica del script `fase4_alertas_criticas.py`. Para mantener el notebook breve, importamos sus funciones.

In [ ]:
import sys
sys.path.append("..")
from notebooks.fase4_alertas_criticas import construir_html, accion_sugerida
print("Funciones importadas desde el script de la Fase 4")

### Celda 5 - Generar y previsualizar el correo HTML

In [ ]:
fecha = datetime.now().strftime("%d/%m/%Y %H:%M")
html = construir_html(criticos, len(df), fecha)
display(HTML(html))

### Celda 6 - Guardar el correo (modo simulado)

In [ ]:
nombre = "alerta_criticos_" + datetime.now().strftime("%Y%m%d") + ".html"
ruta = os.path.join("../output", nombre)
with open(ruta, "w", encoding="utf-8") as f:
    f.write(html)
print("Para   :", DESTINATARIO)
print("Asunto :", ASUNTO)
print("Guardado:", ruta)